In [1]:
import pandas as pd
import numpy as np  
import scanpy as sc
import anndata as ad

In [2]:
# Divides a row by its sum and sclaes it to a reference value
## Input: Frow to be sclaed, reference value
## Output: scaled row

def normalize_vector(v, scale_factor):
    return (v / np.sum(v)) * scale_factor

In [3]:
# Normalizes given cell by features matrix to remove sequencing depth irregularities
## Input: Matrix to be normalized and scaled
## Output: Normalized matrix

def normalize_data_matrix(A, scale_factor):
    A_temp = np.apply_along_axis(normalize_vector, 1, A, scale_factor)  
    A_temp = np.apply_along_axis(np.log1p, 1, A_temp)
    return A_temp

In [4]:
# -------------------------
# 1) Read ADT counts
# -------------------------
data_protein = pd.read_csv('cleaned_adt_tea_seq.csv', index_col=0)
A_counts = data_protein.to_numpy(dtype='float64')   # (n_cells x n_proteins)
adt_names = np.array(data_protein.columns)

# -------------------------
# 2) Per-cell CLR
# -------------------------
## Determining the factor to scale the data 
scale_factor_A = round(np.median(A_counts.sum(axis = 1)), -3)
A_cpm = normalize_data_matrix(A_counts, scale_factor_A)  

U_cpm, S_cpm, Vt_cpm = np.linalg.svd(A_cpm, full_matrices=False)
k_est_cpm = 35
n_features_cpm_adt = A_cpm.shape[1]
n_samples_cpm_adt = A_cpm.shape[0]
A_cpm_residual = A_cpm - (U_cpm[:, :k_est_cpm] @ np.diag(S_cpm[:k_est_cpm]) @ Vt_cpm[:k_est_cpm, :])
tau_sq_cpm = np.sum(A_cpm_residual**2) / (n_features_cpm_adt * n_samples_cpm_adt)

A_cpm = A_cpm / np.sqrt(tau_sq_cpm)

# -------------------------
# 3) Feature selection (top-40 variable proteins on CLR)
# -------------------------
var_cpm = A_cpm.var(axis=0)
topk = 40
idx_top = np.argpartition(var_cpm, -topk)[-topk:]
idx_top = idx_top[np.argsort(var_cpm[idx_top])[::-1]]  # sort descending variance for stability

A_wnn = A_cpm[:, idx_top].copy()  # keep a “natural” CLR matrix for WNN / neighbors
names_top = adt_names[idx_top]
A = A_cpm[:, idx_top]


A -= A.mean(axis=0, keepdims=True)
A_wnn -= A_wnn.mean(axis=0, keepdims=True)

k_A = topk


In [5]:
# -------------------------
# 4) Save cleaned ADT matrix
# -------------------------

# Wrap into a DataFrame for column labels
A_df = pd.DataFrame(
    A,
    columns=names_top,
)

# Preserve cell names if available
if hasattr(data_protein, "index"):
    A_df.index = data_protein.index

# Save to CSV
A_df.to_csv("cleaned_adt_normalized.csv", index=True)
print("Saved cleaned ADT matrix to cleaned_adt_normalized.csv")

Saved cleaned ADT matrix to cleaned_adt_normalized.csv


In [6]:
# --- NEW: Save Raw Counts for Selected ADTs ---
A_counts_selected = A_counts[:, idx_top]

# Wrap the selected counts into a DataFrame for column labels and cell indices
A_counts_df = pd.DataFrame(
    A_counts_selected,
    columns=names_top,
)
if hasattr(data_protein, "index"):
    A_counts_df.index = data_protein.index

A_counts_df.to_csv("cleaned_adt_counts.csv", index=True)
print("Saved raw ADT counts for selected proteins to cleaned_adt_counts.csv")

Saved raw ADT counts for selected proteins to cleaned_adt_counts.csv


In [7]:
# --- NEW: Save the list of selected protein names ---
names_df = pd.DataFrame(
    names_top,
    columns=['ProteinName'] # Name the column for clarity
)
names_df.to_csv("final_adt_features_tea_seq.csv", index=False)
print("Saved selected protein names to cleaned_adt_features.csv")

Saved selected protein names to cleaned_adt_features.csv
